In [1]:
import ee

In [2]:
ee.Authenticate()
ee.Initialize()

In [3]:
from EeImageCollections.S1_GRD.S1GRD_MultiOrbitCollection import S1GRD_MultiOrbitCollection
from EeImageCollections.S1_GRD.S1GRD_SingleOrbitCollection import S1GRD_SingleOrbitCollection
import FeatureManipulation.FeatureCollectionUtils as fcu
from read_and_write.csv_handler import CSVHandler

In [4]:
assets = ee.FeatureCollection('projects/ee-zachariasguislain/assets/LeuvenFields')
roi = assets.bounds()
db_collection = S1GRD_MultiOrbitCollection.of(roi, '2024-01-01', '2024-12-31', 'db')


In [5]:
ro_collections = db_collection.split_into_relative_orbit_collections()
ro_collections

{88: <EeImageCollections.S1_GRD.S1GRD_SingleOrbitCollection.S1GRD_SingleOrbitCollection at 0x2c232696150>,
 161: <EeImageCollections.S1_GRD.S1GRD_SingleOrbitCollection.S1GRD_SingleOrbitCollection at 0x2c236e00ce0>,
 37: <EeImageCollections.S1_GRD.S1GRD_SingleOrbitCollection.S1GRD_SingleOrbitCollection at 0x2c236e01190>,
 110: <EeImageCollections.S1_GRD.S1GRD_SingleOrbitCollection.S1GRD_SingleOrbitCollection at 0x2c236e013d0>}

In [6]:
ro_37 = ro_collections[37]
ro_37.filter_border_noise()
ro_37.add_linear_cross_polarization()
ro_37.add_azimuth_and_local_incidence_angle()
ro_37.get_bands()

{'AZI', 'CP', 'LIA', 'VH', 'VV', 'angle'}

In [7]:
ro_37.allocate_band_to_reducer('mean_count_stdev', 'VV')
ro_37.allocate_bands_to_reducer('mean_stdev', {'CP', 'VH'})
ro_37.allocate_bands_to_reducer('mean', {'AZI', 'LIA'})
ro_37.get_variables_after_reduction()

{'AZI_mean',
 'CP_mean',
 'CP_stdDev',
 'LIA_mean',
 'VH_mean',
 'VH_stdDev',
 'VV_count',
 'VV_mean',
 'VV_stdDev'}

In [8]:
assets = fcu.buffer_fields(assets)
assets = fcu.add_centroid_property(assets)

In [9]:
assets.first().propertyNames().getInfo()

['polygon_centroid', 'Usage', 'id', 'system:index', 'Name']

In [10]:
ro_37_reduced = ro_37.reduce(assets, 'id')

In [11]:
from pathlib import Path
csv_writer = CSVHandler(Path("test"))

In [12]:
csv_writer.write(ro_37_reduced)

In [13]:
csv_writer._meta

{'object_id_name': 'id',
 'mother_collection_id': '3a1834b72cda3452c5e4c5b41f178ed3fe8ef5fe068821267a09c9ac455b6eee',
 'variables': ['VH_stdDev',
  'CP_stdDev',
  'VV_stdDev',
  'VV_count',
  'LIA_mean',
  'VH_mean',
  'VV_mean',
  'CP_mean',
  'AZI_mean'],
 'properties': ['Usage', 'system:index', 'polygon_centroid', 'Name'],
 'orbits': [37],
 'orbit_collection_names': {'37': 'S1_RO_37_Pass_DES_Year_2024'},
 'Orbit_37': ['S1_RO_37_Pass_DES_Year_2024_1of1.csv']}

In [14]:
reduced_collections = set()
csv_io = CSVHandler(Path("test2"))
assets = fcu.add_centroid_property(assets)
ro_collections = db_collection.split_into_relative_orbit_collections()
for ro, ro_collection in ro_collections.items():
    ro_collection.add_azimuth_and_local_incidence_angle()
    ro_collection.filter_border_noise()
    ro_collection.allocate_bands_to_reducer('mean_stdev', {'VV', 'VH', 'LIA', 'AZI'})
    reduced_collections.add(ro_collection.reduce(assets, 'id'))
    print("reduced")

for collection in reduced_collections:
    csv_io.write(collection)
    print("written")


reduced
reduced
reduced
reduced
written
written
written
written


KeyError: '37'